## Imports

In [2]:
import os
import sys
import requests
import json
import time
import torch

from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path
sys.path.insert(0, str(Path().resolve().parent))
from config import data_config
from preprocessing.preprocess_prices import preprocess_prices

## Load Data

In [ ]:
# Load .env file
load_dotenv()

# Get database URL (Alpha Vantage)
api_key = os.environ.get("AV_API_KEY")
base_url = os.environ.get("AV_BASE_URL")

# Get data
Path('../data').mkdir(parents=True, exist_ok=True)
for ticker in data_config.TICKERS:
    url = f"{base_url}function={data_config.AV_FUNCTION}&symbol={ticker}&apikey={api_key}"
    r = requests.get(url)
    data = r.json() 

    # Save data in JSON file
    with open(f'../data/daily_data_{ticker}.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    # Wait
    time.sleep(3)

    print(f"Daily {ticker} data saved to ../data/daily_data_{ticker}.json")


In [3]:
closing_prices_dict = {}

for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/daily_data_{ticker}.json', 'r') as f:
        data = json.load(f)
    
    # Extract time series
    time_series = data['Time Series (Daily)']
    
    # Get closing prices sorted by date (oldest to newest)
    dates = sorted(time_series.keys())
    closing_prices = torch.tensor([float(time_series[date]['4. close']) for date in dates])
    
    # Store in dictionary
    closing_prices_dict[ticker] = closing_prices
    
    # Print summary
    print(f"{ticker}:")
    print(f"  {len(closing_prices)} closing prices")
    print(f"  Date range: {dates[0]} to {dates[-1]}")
    print(f"  Price range: ${closing_prices.min():.2f} to ${closing_prices.max():.2f}")
    print(f"  Array shape: {closing_prices.shape}")
    print()


NVDA:
  100 closing prices
  Date range: 2025-09-12 to 2026-02-04
  Price range: $170.29 to $207.04
  Array shape: torch.Size([100])

GOOG:
  100 closing prices
  Date range: 2025-09-12 to 2026-02-04
  Price range: $237.49 to $344.90
  Array shape: torch.Size([100])

AAPL:
  100 closing prices
  Date range: 2025-09-12 to 2026-02-04
  Price range: $234.07 to $286.19
  Array shape: torch.Size([100])



## Preprocess Data

In [4]:
training_data = {}

for ticker in data_config.TICKERS:
    closing_prices = closing_prices_dict[ticker]
    training_data[ticker] = preprocess_prices(closing_prices, start=0)

print(training_data)


{'NVDA': (tensor([[-4.2147e-08,  1.4034e+00, -2.8631e-01, -8.5677e-02, -5.5958e-01,
         -1.2979e+00, -6.1794e-01,  9.2747e-02],
        [-1.9842e-01,  8.8842e-01, -8.3764e-02, -1.7838e+00,  1.8778e+00,
         -7.6866e-02, -1.0337e+00,  1.2264e+00],
        [-5.2472e-01,  1.6718e+00, -8.4105e-01,  6.4893e-01,  1.0965e+00,
         -5.8182e-01,  3.7735e-01, -9.3859e-01],
        [-2.2058e-01, -4.5612e-01,  1.2773e+00, -2.0708e-02, -7.7101e-01,
         -6.3408e-01, -2.6610e-02,  8.4352e-01],
        [ 4.8270e-01, -1.8520e+00, -1.0969e+00, -6.1997e-01,  3.7995e-02,
         -1.6246e-01, -1.5930e-01,  1.3376e-01],
        [-7.4318e-01,  2.0735e+00,  2.2124e+00, -3.6656e-01,  3.3111e-01,
         -9.9277e-02, -3.6114e-01, -6.3593e-01],
        [ 1.5105e+00, -7.1179e-01, -6.9056e-01, -1.1420e+00,  2.6243e+00,
          1.2087e+00,  1.1095e+00, -1.7182e+00],
        [-6.7591e-01,  1.4150e+00,  9.6878e-01,  2.3824e-01, -9.4945e-01,
         -6.7985e-01,  1.0102e-01,  5.3426e-01]]), tens